# Escha-W2 Codebook Extraction — Google Colab (fallback path)

> **First try `modal_extract.py` instead** — 3-command install, ~10 min, no clicking through Colab. This notebook is the fallback for anyone who can't run Modal.

**What this does:** extracts the AQLM-style residual codebooks baked into
the `escha` C++ runtime, producing `escha_codebooks_v1.npz` (~4-6 MiB) that
the MLX port loads at runtime.

**Runtime required:** GPU (T4 free tier is fine, ~30-45 min).
Go to **Runtime → Change runtime type → T4 GPU** if you haven't already.

**No model weights needed** — the codebooks live inside the runtime `.so`
as compile-time constants; we probe the operator with zero inputs. Total
download is ~200 MB (the escha wheel + torch upgrade), NOT 12 GB.

**Fixes baked in vs the original:**
- Auto-detects the correct probe layout (was hardcoded in v1 and drifted on layout changes)
- Introspects the escha module for a native codebook accessor before probing (may skip the 30-min sweep entirely if one exists)
- Loud `SUCCESS` / `FAILURE` at the end, with a nonzero-row sanity check

Click **Runtime → Run all** and wait. The final cell downloads the
`.npz` to your computer.

## Step 1 — Sanity check GPU + Python version

In [ ]:
import sys, subprocess, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True).strip()
    print('GPU:', out)
except Exception as e:
    print('nvidia-smi failed:', e)
    print('*** FAILURE: Set Runtime → Change runtime type → GPU (T4), then Run all again. ***')
    raise SystemExit(1)

if sys.version_info[:2] != (3, 12):
    print(f'*** FAILURE: The escha wheel is cp312-only; Colab currently has Python {sys.version_info[:2]}. ***')
    print('    Options: (a) use the modal_extract.py path (ships its own py3.12 container),')
    print('             (b) use the RunPod fallback with a py3.12 image,')
    print('             (c) install python3.12 in Colab manually: !apt install -y python3.12 python3.12-venv')
    raise SystemExit(1)


## Step 2 — Install torch 2.9 (CUDA 12.8) + huggingface_hub

Colab preinstalls torch, but the escha wheel is linked against libtorch 2.9. We upgrade to keep ABI compatibility. This takes ~2 minutes.

In [ ]:
%pip install -q --upgrade "torch==2.9.*" --index-url https://download.pytorch.org/whl/cu128
%pip install -q --upgrade "huggingface_hub[cli]" numpy safetensors


In [ ]:
# Verify torch reloaded correctly + CUDA works.
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA not available — set runtime to GPU.'
print('device:', torch.cuda.get_device_name(0))
assert torch.__version__.startswith('2.9'), (
    f'expected torch 2.9.x, got {torch.__version__}. Restart the runtime and re-run.'
)


## Step 3 — Download the escha runtime wheel from HuggingFace

Only the wheel is needed (not the 12 GB Escha-W2 model weights). ~150 MB. Idempotent: re-runs are cached.

In [ ]:
import os, glob, subprocess
os.makedirs('/content/escha', exist_ok=True)
if not glob.glob('/content/escha/sglang/escha-*.whl'):
    subprocess.check_call([
        'hf', 'download',
        'EschaLabs/escha-runtime-qwen3moe',
        '--include', 'sglang/*',
        '--local-dir', '/content/escha',
    ])
wheel = glob.glob('/content/escha/sglang/escha-*.whl')
assert wheel, 'wheel not found after download'
print('wheel:', wheel[0])


In [ ]:
# Install the wheel. If it fails on Python-version mismatch, Colab has
# probably moved off Python 3.12 — fall back to modal_extract.py or RunPod.
%pip install -q /content/escha/sglang/escha-*.whl


In [ ]:
# Load escha and confirm the op is registered.
import torch, escha  # noqa — importing registers torch.ops.escha.*
print('escha module:', escha.__file__)
print('escha version:', getattr(escha, '__version__', '?'))
print('torch.ops.escha ops:', [o for o in dir(torch.ops.escha) if not o.startswith('_')])
op = torch.ops.escha.escham_reconstruct
print('op ready:', op)


## Step 3.5 — Introspect escha for a native codebook accessor

If EschaLabs ships a `dump_codebooks()`, `get_codebooks()`, or similar, use it and skip the 30-min probe. This is a **massive** speedup if it works.

In [ ]:
import escha, torch

escha_attrs = [a for a in dir(escha) if not a.startswith('_')]
ops_names   = [n for n in dir(torch.ops.escha) if not n.startswith('_')]
print('escha module attrs:', escha_attrs)
print('torch.ops.escha ops:', ops_names)

hints = ('codebook', 'lut', 'lattice', 'dump', 'table', 'cb_a', 'cb_b', 'cb_c')
candidates = []
for src, names, root in (('escha', escha_attrs, escha),
                          ('torch.ops.escha', ops_names, torch.ops.escha)):
    for n in names:
        if any(w in n.lower() for w in hints):
            candidates.append((src, n, root))
            print(f'  candidate: {src}.{n}')

# Try to reach the reference python impl bundled inside the wheel
try:
    from escha import transform as _et
    print('escha.transform attrs:', [a for a in dir(_et) if not a.startswith('_')])
except Exception as e:
    print('escha.transform unavailable:', e)

if not candidates:
    print('\nno native accessor found — will use the functional probe path (~30 min).')


## Step 4 — Auto-detect probe layout, then extract K=2 and K=3

Op signature: `op(code, in_features, out_features, K, cbA, mul1) -> (in_p, out_p) fp16`. Code shape: `(in_p/16, out_p/16, 16*K)`. Kernel wants one of (in_p, out_p) divisible by 128. Exact axis convention has drifted across drafts, so we try every plausible layout and lock the one where perturbing `probe.view(-1)[0]` changes exactly one 16-wide slice of the output.

In [ ]:
import time, numpy as np, torch, escha  # noqa

op = torch.ops.escha.escham_reconstruct
DEVICE = 'cuda'

layouts_to_try = [
    ((1,  8, '16K'),  16, 128),
    ((8,  1, '16K'), 128,  16),
    ((8,  8, '16K'), 128, 128),
    ((1, 16, '16K'),  16, 256),
    ((16, 1, '16K'), 256,  16),
]

def _instantiate(shape_desc, K):
    return tuple(16 * K if x == '16K' else x for x in shape_desc)

working_layout = None
for shape_desc, in_f, out_f in layouts_to_try:
    for K in (2, 3):
        probe_shape = _instantiate(shape_desc, K)
        try:
            p0 = torch.zeros(probe_shape, dtype=torch.int16, device=DEVICE)
            w0 = op(p0, in_f, out_f, K, True, False)
        except Exception as e:
            print(f'  {probe_shape} in={in_f} out={out_f} K={K}: baseline raised {type(e).__name__}')
            continue
        p1 = torch.zeros(probe_shape, dtype=torch.int16, device=DEVICE); p1.view(-1)[0] = 1
        try:
            w1 = op(p1, in_f, out_f, K, True, False)
        except Exception as e:
            print(f'  {probe_shape} in={in_f} out={out_f} K={K}: perturbed raised {type(e).__name__}')
            continue
        diff = (w1 - w0).float().abs()
        if diff.ndim >= 2:
            n_changed = int((diff > 1e-6).any(dim=-1).sum().item())
        else:
            n_changed = int((diff > 1e-6).sum().item())
        print(f'  {probe_shape} in={in_f} out={out_f} K={K}: out.shape={tuple(w1.shape)} rows_changed={n_changed}')
        if n_changed == 1 and working_layout is None:
            working_layout = (shape_desc, in_f, out_f)
            print(f'    ^ locked layout')

if working_layout is None:
    print('*** FAILURE: no probe layout produced a clean single-row response. ***')
    print('    The escha op shape convention may have changed. Open a HF discussion:')
    print('    https://huggingface.co/EschaLabs/escha-runtime-qwen3moe/discussions')
    raise SystemExit(1)

shape_desc, IN_F, OUT_F = working_layout
print(f'\nusing layout {shape_desc} in={IN_F} out={OUT_F}')


In [ ]:
def extract(K: int) -> np.ndarray:
    probe_shape = _instantiate(shape_desc, K)
    cb = np.zeros((65536, 16), dtype=np.float16)
    p0 = torch.zeros(probe_shape, dtype=torch.int16, device=DEVICE)
    w0 = op(p0, IN_F, OUT_F, K, True, False).detach().cpu().numpy()
    baseline_row = w0.reshape(-1, 16)[0].copy()
    cb[0] = baseline_row
    t0 = time.time()
    for i in range(1, 65536):
        probe = torch.zeros(probe_shape, dtype=torch.int16, device=DEVICE)
        probe.view(-1)[0] = i
        w = op(probe, IN_F, OUT_F, K, True, False).detach().cpu().numpy()
        cb[i] = w.reshape(-1, 16)[0] - baseline_row
        if i % 4096 == 0:
            elapsed = time.time() - t0
            eta = elapsed * (65536 - i) / i
            print(f'  K={K}: {i}/65536 ({100*i/65536:.1f}%)  elapsed={elapsed:.0f}s  ETA={eta:.0f}s', flush=True)
    print(f'  K={K}: done in {time.time()-t0:.0f}s')
    return cb

print('Extracting K=2 codebook...')
cb_K2 = extract(2)
print('Extracting K=3 codebook...')
cb_K3 = extract(3)

nz2 = int((cb_K2 != 0).any(axis=1).sum())
nz3 = int((cb_K3 != 0).any(axis=1).sum())
print(f'\ncb_K2 nonzero rows: {nz2}/65536')
print(f'cb_K3 nonzero rows: {nz3}/65536')
if nz2 < 60000 or nz3 < 60000:
    print('*** WARNING: unusually few nonzero rows. The probe layout may still be wrong. ***')
    print('    Save the file anyway (for inspection) but treat it as suspect.')
else:
    print('SUCCESS: nonzero-row counts look healthy.')


## Step 5 — Save codebooks in BOTH npz and safetensors formats

In [ ]:
import numpy as np
from safetensors.numpy import save_file

out_npz = '/content/escha_codebooks_v1.npz'
out_st  = '/content/escha_codebooks_v1.safetensors'

np.savez_compressed(out_npz, cb_A_K2=cb_K2, cb_A_K3=cb_K3)
save_file({'cb_A_K2': cb_K2, 'cb_A_K3': cb_K3}, out_st)

import os
for p in (out_npz, out_st):
    print(f'{p}  ({os.path.getsize(p)/1024/1024:.2f} MiB)')


## Step 6 — Download to your computer

Both files. Send `escha_codebooks_v1.npz` back to the person who gave you this notebook — that's the one the MLX port loads.

In [ ]:
from google.colab import files
files.download('/content/escha_codebooks_v1.npz')
files.download('/content/escha_codebooks_v1.safetensors')
print('If downloads did not start, click the folder icon in the left sidebar,')
print('right-click each file, and choose Download.')
